In [2]:
from dotenv import load_dotenv
load_dotenv()

import os

from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    FewShotPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser
)

from langchain_core.runnables import (
    RunnableParallel,
    RunnableLambda,
    RunnablePassthrough
)

llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    max_tokens=200
)

print("LLM Ready ✅")

LLM Ready ✅


# Exercise 1.1 - Translation Chain

### Objective
Build a chain that:

1. Detects the language of the input text
2. Translates it to English (if required)
3. Generates a concise summary

### Concepts Used
- ChatPromptTemplate
- LCEL Chaining
- StrOutputParser
- Multi-step Workflow

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

detect_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Detect the language of the text. Reply with only the language name."),
    ("human",
     "{text}")
])

detect_chain = detect_prompt | llm | StrOutputParser()

translate_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Translate the following {source_language} text to English. "
        "If the text is already English, return it unchanged."
    ),
    ("human", "{text}")
])

translate_chain = translate_prompt | llm | StrOutputParser()

summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Summarize the following text in one concise sentence."
    ),
    ("human", "{text}")
])

summary_chain = summary_prompt | llm | StrOutputParser()

def translation_workflow(text):
    
    language = detect_chain.invoke({
        "text": text
    }).strip()
    
    translated_text = translate_chain.invoke({
        "text": text,
        "source_language": language
    })
    
    summary = summary_chain.invoke({
        "text": translated_text
    })
    
    return {
        "language": language,
        "translated_text": translated_text,
        "summary": summary
    }
    

In [4]:
sample_text = "Bonjour, je voudrais réserver une table pour deux personnes ce soir à 20 heures."

result = translation_workflow(sample_text)

print("Detected Language:", result["language"])
print("\nTranslated Text:")
print(result["translated_text"])

print("\nSummary:")
print(result["summary"])

Detected Language: French

Translated Text:
Hello, I would like to reserve a table for two people this evening at 8 PM.

Summary:
I appreciate your interest, but I should clarify that I'm an AI assistant and don't have the ability to make restaurant reservations. 

To book a table, I'd recommend:
- **Calling the restaurant directly**
- **Using reservation apps** like OpenTable, Resy, or Yelp
- **Visiting the restaurant's website** for their booking system
- **Checking Google Maps** for reservation options

If you let me know which restaurant you're interested in, I'd be happy to help you find their contact information or suggest how to reach them!


# Exercise 1.2 - Product Review Analyzer

### Objective
Analyze customer reviews using RunnableParallel.

Extract:
- Sentiment
- Pros
- Cons
- Predicted Rating

### Concepts Used
- RunnableParallel
- Parallel Processing
- Prompt Templates

In [5]:
sample_review = """
I bought this laptop 3 months ago. The screen quality is amazing and the battery lasts all day.
However, the keyboard feels a bit mushy and it runs hot when gaming. For the price, it's decent
but not exceptional. Would recommend for casual users but not power users.
"""

In [6]:
sentiment_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Analyze the sentiment of this review. Respond only with POSITIVE, NEGATIVE, or NEUTRAL."
    ),
    ("human", "{review}")
])

pros_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Extract the key advantages or pros mentioned in this review."
    ),
    ("human", "{review}")
])

cons_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Extract the key disadvantages or cons mentioned in this review."
    ),
    ("human", "{review}")
])

rating_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Predict a star rating from 1 to 5 based on this review. Reply only with the number."
    ),
    ("human", "{review}")
])

In [7]:
review_analyzer = RunnableParallel(
    
    sentiment=(
        sentiment_prompt
        | llm
        | StrOutputParser()
    ),
    
    pros=(
        pros_prompt
        | llm
        | StrOutputParser()
    ),
    
    cons=(
        cons_prompt
        | llm
        | StrOutputParser()
    ),
    
    rating=(
        rating_prompt
        | llm
        | StrOutputParser()
    )
)

In [8]:
review_result = review_analyzer.invoke({
    "review": sample_review
})

print("=" * 60)
print("PRODUCT REVIEW ANALYSIS")
print("=" * 60)

for key, value in review_result.items():
    print(f"\n{key.upper()}:")
    print(value)

PRODUCT REVIEW ANALYSIS

SENTIMENT:
NEUTRAL

The review contains both positive elements (screen quality, battery life) and negative elements (mushy keyboard, heat during gaming), along with a qualified recommendation. The overall tone is balanced rather than leaning strongly toward satisfaction or dissatisfaction.

PROS:
# Key Advantages (Pros):

1. **Amazing screen quality** - The display is a standout feature
2. **All-day battery life** - Excellent for portability and mobility
3. **Good value for the price** - Decent performance relative to cost
4. **Suitable for casual users** - Reliable for everyday tasks

CONS:
# Key Disadvantages/Cons:

1. **Mushy keyboard** - The keyboard lacks a responsive feel and tactile feedback

2. **Runs hot during gaming** - Thermal management issues when under heavy load

3. **Not ideal for power users** - Limited performance or capabilities for demanding tasks

4. **Price-to-value ratio** - Described as "decent but not exceptional," suggesting it may be

# Exercise 1.3 - Legal Document Processing Pipeline

### Objective

Process legal documents and extract:

1. Key dates
2. Parties involved
3. Document type
4. Main obligations
5. Potential risks

### Concepts Used
- RunnableParallel
- Prompt Chaining
- Information Extraction
- Legal Document Analysis

In [9]:
sample_document = """
SERVICE AGREEMENT

This Agreement is entered into as of January 15, 2024, between:
- TechCorp Inc. ("Service Provider"), located at 123 Tech Street, San Francisco, CA
- ClientCo LLC ("Client"), located at 456 Business Ave, New York, NY

TERMS:
1. Service Provider agrees to deliver software development services for a period of 12 months.
2. Client agrees to pay $50,000 per month, due on the 1st of each month.
3. Late payments will incur a 5% penalty fee.
4. Either party may terminate with 30 days written notice.
5. All intellectual property created shall belong to the Client.
6. Service Provider shall maintain confidentiality for 5 years after termination.

Signed by both parties on the date first written above.
"""

In [11]:
dates_parties_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Extract:
        - Important dates
        - Organizations/parties involved

        Format clearly.
        """
    ),
    ("human", "{document}")
])

document_type_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Identify the legal document type."
    ),
    ("human", "{document}")
])

obligations_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Summarize the major obligations of all parties."
    ),
    ("human", "{document}")
])

risk_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Identify potential risks, concerns,
        liabilities, penalties, and clauses that
        require special attention.
        """
    ),
    ("human", "{document}")
])

In [12]:
legal_analysis_chain = RunnableParallel(
    
    parties_and_dates=(
        dates_parties_prompt
        | llm
        | StrOutputParser()
    ),
    
    document_type=(
        document_type_prompt
        | llm
        | StrOutputParser()
    ),
    
    obligations=(
        obligations_prompt
        | llm
        | StrOutputParser()
    ),
    
    risks=(
        risk_prompt
        | llm
        | StrOutputParser()
    )
)

In [13]:
legal_result = legal_analysis_chain.invoke({
    "document": sample_document
})

print("=" * 70)
print("LEGAL DOCUMENT ANALYSIS")
print("=" * 70)

print("\nDOCUMENT TYPE")
print("-" * 70)
print(legal_result["document_type"])

print("\nDATES AND PARTIES")
print("-" * 70)
print(legal_result["parties_and_dates"])

print("\nMAIN OBLIGATIONS")
print("-" * 70)
print(legal_result["obligations"])

print("\nRISKS AND CONCERNS")
print("-" * 70)
print(legal_result["risks"])


LEGAL DOCUMENT ANALYSIS

DOCUMENT TYPE
----------------------------------------------------------------------
# Legal Document Type Identification

## Document Type: **SERVICE AGREEMENT**

### Classification Details:

**Primary Category:** Commercial Contract

**Specific Type:** Professional Services Agreement (Software Development Services Agreement)

### Key Characteristics Identifying This Document:

1. **Parties Involved:** Service provider (TechCorp Inc.) and client (ClientCo LLC)
2. **Scope of Work:** Defined services (software development)
3. **Financial Terms:** Clear payment structure ($50,000/month)
4. **Duration:** Specified term (12 months)
5. **Standard Clauses:**
   - Payment terms and penalties
   - Termination conditions
   - Intellectual property rights
   - Confidentiality obligations

### Related Document Categories:
- Commercial Contract
- B2B (Business-to-Business) Agreement
- Independent Contractor/Vendor Agreement
- Professional Services

DATES AND PARTIES
------